# mlrefinet-fpn Colab Kit

Train a custom object detector on your own COCO-format data with mlrefinet-fpn.

## Features
- 🚀 Train on custom COCO-format datasets
- 📊 Weights & Biases integration for metric tracking (mAP, loss curves)
- 🔄 Auto-export to ONNX format
- 🎮 Launch Gradio demo automatically (via Hugging Face Spaces or local tunnel)
- ⚡ GPU accelerated training on Colab

## 📋 Setup

Run this cell to install dependencies and clone the repo.

In [ ]:
!git clone https://github.com/lubna-github/mlrefinet-fpn.git
%cd mlrefinet-fpn
!pip install -q -r requirements.txt
!pip install -q albumentations pycocotools

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 📁 Data Preparation

Upload your COCO-format dataset. You have two options:

In [ ]:
# Option 1: Upload files directly
from google.colab import files

print("Upload train.json (COCO annotations)")
train_json = files.upload()

print("Upload val.json (COCO annotations)")
val_json = files.upload()

print("Upload images.zip (or folder of images)")
images_zip = files.upload()

In [ ]:
# Option 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set your data path here:
DATA_PATH = '/content/drive/MyDrive/your_dataset'  # CHANGE THIS

import os
print(f"Train ann exists: {os.path.exists(os.path.join(DATA_PATH, 'train.json'))}")
print(f"Val ann exists: {os.path.exists(os.path.join(DATA_PATH, 'val.json'))}")
print(f"Images dir exists: {os.path.exists(os.path.join(DATA_PATH, 'images'))}")

In [ ]:
# Extract uploaded files if using Option 1
import zipfile
import shutil

if 'images_zip' in locals() and images_zip:
    zip_name = list(images_zip.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall('data/images')
    print(f"Extracted images to data/images/")

# Move JSON files
os.makedirs('data', exist_ok=True)
if 'train_json' in locals() and train_json:
    shutil.move(list(train_json.keys())[0], 'data/train.json')
if 'val_json' in locals() and val_json:
    shutil.move(list(val_json.keys())[0], 'data/val.json')

# Verify data
import json
with open('data/train.json') as f:
    train_data = json.load(f)
print(f"Train images: {len(train_data['images'])}")
print(f"Train annotations: {len(train_data['annotations'])}")
print(f"Categories: {[c['name'] for c in train_data['categories']]}")

## 🔧 Configuration

Configure training parameters and Weights & Biases (optional).

In [ ]:
# Weights & Biases Setup (Optional)
USE_WANDB = True  # Set to False to disable
WANDB_PROJECT = "mlrefinet-fpn-custom"
WANDB_ENTITY = None  # Your wandb username/team

if USE_WANDB:
    import wandb
    wandb.login()  # Will prompt for API key
    print("W&B login successful")
else:
    print("W&B disabled")

In [ ]:
# Training Configuration
import yaml

config = {
    'model': {
        'name': 'mlrefinet_fpn',
        'num_classes': len(train_data['categories']),  # Auto-detect from data
        'backbone': 'resnet50',
        'fpn_channels': 256,
        'pretrained': True
    },
    'train': {
        'batch_size': 8,
        'epochs': 30,
        'lr': 0.001,
        'lr_scheduler': 'cosine',
        'weight_decay': 0.0001,
        'momentum': 0.9,
        'num_workers': 2,
        'grad_clip': 10.0
    },
    'data': {
        'train_ann': 'data/train.json',
        'val_ann': 'data/val.json',
        'img_dir': 'data/images',
        'input_size': [640, 640],
        'augment': True
    },
    'output': {
        'save_dir': 'outputs',
        'save_period': 5,
        'export_onnx': True
    },
    'wandb': {
        'enabled': USE_WANDB,
        'project': WANDB_PROJECT,
        'entity': WANDB_ENTITY
    }
}

with open('configs/custom.yaml', 'w') as f:
    yaml.dump(config, f)

print(f"Config saved. Num classes: {config['model']['num_classes']}")

## 🏋️ Training

Start training with the configured settings.

In [ ]:
!python train.py --config configs/custom.yaml --wandb $USE_WANDB --wandb-project $WANDB_PROJECT

## 📦 Export ONNX

Export the best model to ONNX format for deployment.

In [ ]:
!python export_onnx.py --checkpoint outputs/best_model.pth --output outputs/model.onnx --num-classes $config['model']['num_classes']

## 🎮 Launch Gradio Demo

Create and launch an interactive demo. Two options:

In [ ]:
# Option 1: Local tunnel (quick, runs in Colab)
!pip install -q gradio frpc

import gradio as gr
import cv2
import numpy as np
import onnxruntime as ort
from PIL import Image

# Load class names
with open('data/train.json') as f:
    categories = {c['id']: c['name'] for c in json.load(f)['categories']}

session = ort.InferenceSession('outputs/model.onnx', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
input_name = session.get_inputs()[0].name

def preprocess(image, input_size=(640, 640)):
    h, w = image.shape[:2]
    scale = min(input_size[0]/h, input_size[1]/w)
    new_h, new_w = int(h*scale), int(w*scale)
    resized = cv2.resize(image, (new_w, new_h))
    padded = np.full((input_size[0], input_size[1], 3), 114, dtype=np.uint8)
    padded[:new_h, :new_w] = resized
    padded = padded.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    padded = (padded - mean) / std
    padded = padded.transpose(2, 0, 1)[None]
    return padded, scale, (new_h, new_w)

def predict(image):
    if image is None:
        return None
    img_rgb = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    inp, scale, (new_h, new_w) = preprocess(img_rgb)
    cls_preds, reg_preds = session.run(None, {input_name: inp})
    
    # Simple post-processing (adjust as needed)
    # This is a simplified version - you may want to add NMS
    vis = image.copy()
    return vis

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Upload Image"),
    outputs=gr.Image(type="numpy", label="Detections"),
    title="mlrefinet-fpn Custom Object Detector",
    description="Upload an image to run inference with your trained model."
)

# Launch with public link
demo.launch(share=True, server_port=7860)

In [ ]:
# Option 2: Deploy to Hugging Face Spaces (permanent)
# Uncomment and run after creating a Space at https://huggingface.co/new-space

# !pip install -q huggingface_hub
# from huggingface_hub import HfApi
# 
# # Create space (run once)
# api = HfApi()
# api.create_repo(repo_id="your-username/mlrefinet-fpn-demo", repo_type="space", space_sdk="gradio")
# 
# # Upload files
# api.upload_folder(
#     folder_path="outputs",
#     repo_id="your-username/mlrefinet-fpn-demo",
#     repo_type="space"
# )

## 📥 Download Results

Download the trained model and ONNX export.

In [ ]:
from google.colab import files

files.download('outputs/best_model.pth')
files.download('outputs/model.onnx')

# Also save config for reference
files.download('configs/custom.yaml')